# Método dos Gradientes Conjugados — 2º Grau e 3º Grau

Este notebook apresenta uma versão refatorada, limpa e comentada do algoritmo desenvolvido no meu **TCC**, além de testes práticos para:
1. **Funções Quadráticas (2º Grau)** em $\mathbb{R}^2$ ($2 \times 2$) e $\mathbb{R}^3$ ($3 \times 3$);
2. **Polinômios de 3º Grau (Funções Cúbicas)** usando a extensão não-linear (Fletcher-Reeves / Newton-CG).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. Funções Base: Função Objetivo e Gradiente (2º Grau)
Para uma função quadrática $f(x) = \frac{1}{2} x^T A x + b^T x + c$, o gradiente é $\nabla f(x) = Ax + b$:

In [ ]:
def funcao_quadratica(x: np.ndarray, A: np.ndarray, b: np.ndarray, c: float = 0.0) -> float:
    """Calcula o valor escalar f(x) = (1/2) * x^T A x + b^T x + c."""
    return float(0.5 * (x @ A @ x) + (b @ x) + c)


def gradiente(x: np.ndarray, A: np.ndarray, b: np.ndarray) -> np.ndarray:
    """Calcula o vetor gradiente ∇f(x) = A x + b."""
    return A @ x + b

## 2. Algoritmo para Funções de 2º Grau (Quadráticas)
Em funções de 2º grau, a matriz de curvatura (Hessiana $A$) é constante e o algoritmo converge em **no máximo $n$ passos**.

In [ ]:
def gradiente_conjugado(A: np.ndarray, b: np.ndarray, x0: np.ndarray, tol: float = 1e-8, max_iter: int = 100):
    x = x0.astype(float).copy()
    historico = [x.copy()]

    # Passo 0: Gradiente inicial e primeira direção (máxima descida)
    g = gradiente(x, A, b)
    d = -g

    for k in range(max_iter):
        norma_g = np.linalg.norm(g)
        print(f"Iteração {k:2d} | x = {np.round(x, 6)} | ||grad|| = {norma_g:.2e}")

        if norma_g < tol:
            print(f"\n✅ Ponto de mínimo encontrado em {k} iterações!")
            break

        Ad = A @ d
        curvatura = d @ Ad  # d^T A d

        # 1. Tamanho ótimo do passo (t_k)
        t = -(g @ d) / curvatura

        # 2. Novo ponto (x_{k+1})
        x = x + t * d
        historico.append(x.copy())

        # 3. Novo gradiente (g_{k+1})
        g_novo = gradiente(x, A, b)

        # 4. Coeficiente de conjugação (Beta_k)
        beta = (Ad @ g_novo) / curvatura

        # 5. Nova direção conjugada (d_{k+1})
        d = -g_novo + beta * d
        g = g_novo

    return x, np.array(historico)

### Teste 2.1: Função de 2º Grau Diagonal (Dados Originais do TCC)
Matriz $A = 15I$ (converge em **1 passo**):

In [ ]:
A_tcc = np.array([[15.0, 0.0], [0.0, 15.0]])
b_tcc = np.array([8.0, 3.0])
x0_tcc = np.array([10.0, 10.0])

x_min_tcc, _ = gradiente_conjugado(A_tcc, b_tcc, x0_tcc)
print("Ponto mínimo:", x_min_tcc)

### Teste 2.2: Função de 2º Grau em 3 Variáveis ($3 \times 3$ — Converge em 3 passos)

In [ ]:
A_3d = np.array([
    [4.0, 1.0, 0.0],
    [1.0, 3.0, 1.0],
    [0.0, 1.0, 2.0],
])
b_3d = np.array([-5.0, -5.0, -3.0])  # Solução exata: [1, 1, 1]
x0_3d = np.array([10.0, -5.0, 8.0])

x_min_3d, _ = gradiente_conjugado(A_3d, b_3d, x0_3d)
print("Ponto mínimo encontrado:", x_min_3d)

## 3. Extensão e Testes para Polinômios de 3º Grau (Funções Cúbicas)
Em uma função polinomial de **3º grau**, o gradiente $\nabla f(x)$ é de **2º grau** e a matriz de segundas derivadas (Hessiana $H(x) = \nabla^2 f(x)$) não é mais fixa — ela depende da posição atual $x$.

Por exemplo, considere o polinômio de 3º grau para $x_1, x_2 > 0$:
$$f(x_1, x_2) = \frac{1}{3}(x_1^3 + x_2^3) + 2x_1^2 + \frac{3}{2}x_2^2 - 5x_1 - 4x_2$$
* **Gradiente:** $\nabla f(x) = [x_1^2 + 4x_1 - 5, \; x_2^2 + 3x_2 - 4]^T$ (que se anula em $x^* = [1, 1]^T$)
* **Hessiana:** $H(x) = \begin{bmatrix} 2x_1 + 4 & 0 \\ 0 & 2x_2 + 3 \end{bmatrix}$

In [ ]:
def gradiente_conjugado_terceiro_grau(grad_f, hess_f, x0: np.ndarray, tol: float = 1e-8, max_iter: int = 100):
    x = x0.astype(float).copy()
    n = len(x)
    historico = [x.copy()]

    g = grad_f(x)
    d = -g

    for k in range(max_iter):
        norma_g = np.linalg.norm(g)
        print(f"Iteração {k:2d} | x = {np.round(x, 6)} | ||grad|| = {norma_g:.2e}")

        if norma_g < tol:
            print(f"\n✅ Ponto de mínimo do polinômio de 3º grau encontrado em {k} iterações!")
            break

        # A Hessiana H(x) é recalculada no ponto atual x
        H = hess_f(x)
        curvatura = d @ H @ d

        # 1. Passo de Newton-CG usando a curvatura local
        t = -(g @ d) / curvatura

        # 2. Atualiza x
        x = x + t * d
        historico.append(x.copy())

        # 3. Novo gradiente
        g_novo = grad_f(x)

        # 4. Coeficiente de Fletcher-Reeves com reinício a cada n passos
        beta = 0.0 if (k + 1) % n == 0 else (g_novo @ g_novo) / (g @ g)

        # 5. Nova direção conjugada
        d = -g_novo + beta * d
        g = g_novo

    return x, np.array(historico)


# Definindo o gradiente e a Hessiana do polinômio de 3º grau
def grad_cubico(x: np.ndarray) -> np.ndarray:
    return np.array([x[0]**2 + 4.0*x[0] - 5.0, x[1]**2 + 3.0*x[1] - 4.0])

def hess_cubico(x: np.ndarray) -> np.ndarray:
    return np.array([[2.0*x[0] + 4.0, 0.0], [0.0, 2.0*x[1] + 3.0]])

x_min_cubico, traj_cubico = gradiente_conjugado_terceiro_grau(grad_cubico, hess_cubico, np.array([6.0, 6.0]))
print("Ponto mínimo encontrado:", x_min_cubico)

## 4. Visualização Gráfica das Trajetórias (2º Grau vs. 3º Grau)

In [ ]:
A_elipse = np.array([[6.0, 2.0], [2.0, 3.0]])
b_elipse = np.array([-4.0, 2.0])
x_min_q, traj_q = gradiente_conjugado(A_elipse, b_elipse, np.array([8.0, 8.0]))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: 2º Grau
x1_q, x2_q = np.meshgrid(np.linspace(-2, 9, 150), np.linspace(-4, 9, 150))
Z_q = 0.5 * (A_elipse[0,0]*x1_q**2 + 2*A_elipse[0,1]*x1_q*x2_q + A_elipse[1,1]*x2_q**2) + b_elipse[0]*x1_q + b_elipse[1]*x2_q
axes[0].contour(x1_q, x2_q, Z_q, levels=25, cmap="viridis", alpha=0.7)
axes[0].plot(traj_q[:, 0], traj_q[:, 1], "ro-", linewidth=2, label="Trajetória (2 passos)")
axes[0].plot(x_min_q[0], x_min_q[1], "y*", markersize=14, markeredgecolor="black", label="Mínimo $x^*$")
axes[0].set_title("Função de 2º Grau (Quadrática)")
axes[0].legend()
axes[0].grid(True, linestyle="--", alpha=0.4)

# Gráfico 2: 3º Grau
x1_c, x2_c = np.meshgrid(np.linspace(0, 7, 150), np.linspace(0, 7, 150))
Z_c = (1/3)*(x1_c**3 + x2_c**3) + 2*x1_c**2 + 1.5*x2_c**2 - 5*x1_c - 4*x2_c
axes[1].contour(x1_c, x2_c, Z_c, levels=25, cmap="plasma", alpha=0.7)
axes[1].plot(traj_cubico[:, 0], traj_cubico[:, 1], "bo-", linewidth=2, label="Trajetória Newton-CG")
axes[1].plot(x_min_cubico[0], x_min_cubico[1], "y*", markersize=14, markeredgecolor="black", label="Mínimo $x^* = (1, 1)$")
axes[1].set_title("Função de 3º Grau (Cúbica)")
axes[1].legend()
axes[1].grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()